# FedSwarm — Phase 2.1 centralized ceiling (Colab GPU)

Trains the centralized performance ceiling every FL method in later phases gets measured against: `SimpleCNN` (primary config, 112px) across 5 seeds x {groupnorm, batchnorm}, plus an optional secondary `ResNet-18` @224 table.

**Before running:**
1. **Turn on GPU.** Runtime -> Change runtime type -> Hardware accelerator -> GPU (T4). No account verification needed, unlike Kaggle.
2. **Have `archive (2).zip` (the Brain Tumor MRI dataset, ~164MB) ready to upload** -- the same file used to build the manifest locally. The cell below prompts an upload dialog; point it at that file from your Downloads folder.

This notebook does **not** need `flwr` -- that's only required once the Flower client/server harness exists (Phase 3+). Centralized training is plain PyTorch.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/researchpaper784-alt/ResearchPaper.git"
REPO_DIR = "/content/ResearchPaper"

# Colab sessions can survive a cell re-run (e.g. retrying after an earlier cell
# failed) without wiping /content, so a plain `git clone` here fails with exit
# code 128 ("destination path already exists and is not an empty directory") on a
# rerun. Make this idempotent: pull if it's already a clone of this repo, re-clone
# if the directory exists but isn't (a partial/failed prior clone), else clone.
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
elif os.path.isdir(REPO_DIR):
    subprocess.run(["rm", "-rf", REPO_DIR], check=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

In [ ]:
%cd /content/ResearchPaper

# Colab's base image already has torch/torchvision/numpy/pandas/scikit-learn/
# scipy/matplotlib/pillow/tqdm. Only these are missing for centralized training
# (flwr/flwr-datasets/imagehash/kaggle are not imported by this training path).
!pip install -q omegaconf rich

# Install fedswarm itself (editable, no deps -- everything it needs is already
# satisfied above). Every step below shells out to a fresh interpreter via
# `!python -m fedswarm...` or `subprocess.run(["python", ...])`, and none of those
# inherit this kernel's in-memory sys.path -- so without a real install they all
# fail with `ModuleNotFoundError: No module named 'fedswarm'`. That is what
# silently broke the dataset download, cache build, and training steps on every
# prior Colab run.
!pip install -q -e . --no-deps

## Saving progress across sessions (run this before anything else)

Colab's GPU quota typically runs out before all 10 runs finish, and `/content` is
wiped when the runtime is recycled. The cell below mounts Drive and restores
whatever a previous session saved: finished result JSONs, and the decoded-image
cache.

**If it restores a cache, skip the upload/download cells below entirely.** Once the
cache exists, `run_experiment.py` reads only `manifest.csv` (committed to the repo)
and the cache array -- the raw images are never touched again, so the 164MB
re-upload is only ever needed once.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import shutil

DRIVE = "/content/drive/MyDrive/fedswarm_backup"


def save_progress():
    """Copy finished results + the decoded-image cache to Drive."""
    os.makedirs(DRIVE, exist_ok=True)
    for src, dst in (("results/centralized", "results_centralized"), ("data/processed/cache", "cache")):
        if os.path.isdir(src):
            shutil.copytree(src, f"{DRIVE}/{dst}", dirs_exist_ok=True)
    done = sorted(os.listdir(f"{DRIVE}/results_centralized")) if os.path.isdir(f"{DRIVE}/results_centralized") else []
    print(f"saved to Drive -- {len(done)} finished runs: {done}")


def restore_progress():
    """Bring a previous session's results + cache back into this runtime."""
    for src, dst in (("results_centralized", "results/centralized"), ("cache", "data/processed/cache")):
        if os.path.isdir(f"{DRIVE}/{src}"):
            shutil.copytree(f"{DRIVE}/{src}", dst, dirs_exist_ok=True)
            print(f"restored {dst}")


restore_progress()

## Upload the dataset

**Skip this cell and the next one if the Drive restore above brought back a cache**
-- they only exist to produce that cache in the first place.

Running this cell opens a file picker. Select `archive (2).zip` from your Downloads
folder -- the same Brain Tumor MRI Dataset zip used locally to build `manifest.csv`.
Upload takes a minute or two depending on your connection; the file is ~164MB.


In [ ]:
from google.colab import files

uploaded = files.upload()
zip_name = next(iter(uploaded))
print(f"Uploaded: {zip_name} ({len(uploaded[zip_name]) / 1e6:.1f} MB)")

In [ ]:
import os
import subprocess

os.environ["FEDSWARM_DATA_ROOT"] = "/content/brain-tumor-mri"

# Invoked via subprocess rather than `!python ... --zip "{zip_name}"`. IPython's
# brace interpolation is all-or-nothing per line: it resolves `$NAME` against the
# *Python* namespace, so `$FEDSWARM_DATA_ROOT` (an environment variable, not a
# Python one) raised, and var_expand swallowed the error and passed the entire
# line through untouched -- shipping the literal text `{zip_name}` to argparse.
# An argument list also sidesteps quoting the uploaded filename, which Colab
# renames to things like `archive (1) (1).zip` (spaces and parens included).
proc = subprocess.run(
    ["python", "-m", "fedswarm.data.download",
     "--zip", zip_name,
     "--root", os.environ["FEDSWARM_DATA_ROOT"]],
    capture_output=True, text=True,
)
print(proc.stdout, proc.stderr)
proc.check_returncode()

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected -- check Runtime > Change runtime type before continuing")

## Build the decoded-image cache

Uses the manifest already committed to the repo (`data/processed/manifest.csv`) -- the pseudo-patient-level split from Phase 1.3, with cross-split leakage verified zero. This step is I/O-bound (JPEG decode via PIL), not GPU-bound, and should take well under a minute for 7,200 images.

In [ ]:
import subprocess

# ensure_cache builds only when the array is absent, so this is a no-op when a
# previous session's cache was just restored from Drive (the CLI entry point,
# `python -m fedswarm.data.cache`, always rebuilds unconditionally by design).
# capture_output + print because Jupyter does not display a subprocess's own
# writes to the kernel's file descriptor -- see the training cell for the details.
proc = subprocess.run(
    ["python", "-c", "from fedswarm.data.cache import ensure_cache; ensure_cache(112)"],
    capture_output=True, text=True,
)
print(proc.stdout, proc.stderr)
proc.check_returncode()

## Primary ceiling: SimpleCNN @112, 5 seeds x {groupnorm, batchnorm}

GroupNorm is the FL-relevant default; BatchNorm is run alongside as the A9 confound-control ablation (BatchNorm running statistics aggregate badly under non-IID federated data -- this comparison shows whether that's actually visible here, not just asserted).

In [ ]:
import subprocess
import sys
from pathlib import Path


def run_experiment(config_layers, seed, overrides=()):
    """Run one experiment, streaming the child's output into the notebook.

    Jupyter only captures writes to Python's sys.stdout, not to the kernel's
    underlying file descriptor -- so a plain subprocess.run() shows nothing in the
    cell: no epoch logs, and no traceback when the child dies. Every failure then
    surfaces as a bare CalledProcessError with the real cause invisible. Reading the
    pipe and re-printing it puts both back where they can be seen.
    """
    cmd = ["python", "scripts/run_experiment.py", "--config", *config_layers, "--seed", str(seed)]
    if overrides:
        cmd += ["--override", *overrides]

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        sys.stdout.flush()
    return proc.wait()


CONFIG = ["configs/base.yaml", "configs/model/simple_cnn.yaml", "configs/experiment/centralized.yaml"]

for norm in ["groupnorm", "batchnorm"]:
    for seed in [0, 1, 2, 3, 4]:
        # A result file is written only after a run finishes, so an interrupted run
        # leaves none and is retried, while finished ones are never recomputed.
        if Path(f"results/centralized/simple_cnn_{norm}_{seed}.json").exists():
            print(f"skip norm={norm} seed={seed} (already done)")
            continue

        print(f"\n{'='*20} norm={norm} seed={seed} {'='*20}")
        code = run_experiment(CONFIG, seed, [f"model.norm={norm}"])
        if code != 0:
            raise SystemExit(f"run failed (exit {code}) -- see the traceback above")

        # Checkpoint to Drive after every run: Colab's GPU quota can cut the session
        # off at any point, and /content does not survive a runtime recycle.
        save_progress()

In [ ]:
!python scripts/summarize_centralized.py

## Optional: secondary ResNet-18 @224 table

"Does the ceiling hold with a larger, pretrained backbone." More expensive than the primary sweep (224px + 11.2M params) -- skip this cell entirely to save time/quota if you just need the primary ceiling.

In [ ]:
subprocess.run(
    ["python", "-c", "from fedswarm.data.cache import ensure_cache; ensure_cache(224)"],
    check=True,
)

RESNET_CONFIG = ["configs/base.yaml", "configs/model/resnet18.yaml", "configs/experiment/centralized_resnet18.yaml"]

for seed in [0, 1, 2, 3, 4]:
    if Path(f"results/centralized/resnet18_groupnorm_{seed}.json").exists():
        print(f"skip resnet18 seed={seed} (already done)")
        continue
    print(f"\n{'='*20} resnet18 seed={seed} {'='*20}")
    code = run_experiment(RESNET_CONFIG, seed)
    if code != 0:
        raise SystemExit(f"run failed (exit {code}) -- see the traceback above")
    save_progress()

## Getting results back

This zips `results/centralized/*.json` and triggers a browser download directly -- no output-tab hunting like Kaggle. Send the downloaded zip back for analysis.

In [ ]:
from google.colab import files

!cd results && zip -r /content/centralized_results.zip centralized/
files.download("/content/centralized_results.zip")